###Transform Results Data

1. Read bronze results table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId -> constructor_id, driverId -> driver_id, raceName -> race_name, positionText -> finish_position_text)
4. Rename columns to make them more meaningful (date -> race_date, grid -> grid_position, laps -> completed_laps, number -> car_number, position -> finish_position)
5. Filter out rows where season, round, constructor_id or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns race_name to Title Case
8. Write the transformed data to silver results table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

#####Step 1 - 4 Read bronze results table, select only the required columns and standardise column names

In [0]:
results_df = (
    spark.table(bronze_table)
        .select(
            "season",
            "round",
            "constructorId",
            "driverId",
            "date",
            "raceName", 
            "grid",
            "laps",
            "number",
            "points",
            "position",
            "positionText",
            "status",
            "ingestion_timestamp",
            "source_file"
        )
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "final_position",
            "positionText": "final_position_text",
        })
)

#####Step 5 & 6 Apply Data Quality Checks

In [0]:
results_valid_df = (
    results_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])

)

In [0]:
display(results_df.count() - results_valid_df.count())

#####Step 7 - Transform values of column race_name to Title Case

In [0]:
results_final_df = (results_df
                     .withColumn('race_name', F.initcap(F.col("race_name")))
                    )

#####Step 8 - Write the transformed data to silver results table

In [0]:
(
    results_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))